<a href="https://colab.research.google.com/github/MightyCrimsonX/Crimson-Notebooks/blob/main/Notebooks/MightyCrimson_SimpleAnima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title #📥 Celda 1: Instalar ComfyUI Backend y Descargar Componentes de Anima
#@markdown #Obligatorio: Ingresa tu token de CivitAI
#@markdown Modelos Disponibles: Anima Base, RDBT Anima 1.2
token_civitai = "" #@param {type:"string"}

import pickle, pathlib
pathlib.Path.home().joinpath(".civitai_token.pkl").write_bytes(pickle.dumps(token_civitai))
#@markdown ---

#@markdown ### <font color="#f73100"> **Elige Los modelos a usar** </font>
Anima_v1_Base = True # @param {"type":"boolean"}
RBDTAnima_b12 = False # @param {"type":"boolean"}

import os as _os
import subprocess as _sp
import time as _time
import socket as _socket
from IPython.display import clear_output

print("============================================================")
print("📦 CONFIGURANDO ENTORNO COMFYUI...")
print("============================================================")
%cd /content
# 1. Clonar ComfyUI si no existe
!wget -q https://raw.githubusercontent.com/MightyCrimsonX/Notebook_Scripts/refs/heads/main/scripts/download_magic.py
if not _os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd ComfyUI

# 2. Instalar dependencias esenciales
print("\n⚡ Instalando librerías del sistema...")
!uv pip install transformers accelerate safetensors sentencepiece protobuf
!uv pip install torch==2.9.1 torchvision==0.24.1 torchaudio==2.9.1 xformers==0.0.33.post2 triton==3.5.1 --index-url https://download.pytorch.org/whl/cu128 --no-progress
!uv pip install "numpy==2.0.2" websocket-client tqdm
!uv pip install -r requirements.txt
!pip install aria2

# 3. Asegurar rutas de almacenamiento y descargar componentes de Anima
_os.makedirs("models/diffusion_models", exist_ok=True)
_os.makedirs("models/text_encoders", exist_ok=True)
_os.makedirs("models/vae", exist_ok=True)
_os.makedirs("models/upscale_models", exist_ok=True)
import download_magic
%cd /content/ComfyUI/models/diffusion_models
if Anima_v1_Base:
  %download https://civitai.red/api/download/models/2945208?fileId=2824391
if RBDTAnima_b12:
  %download https://civitai.red/api/download/models/3150130?fileId=3030783

%cd /content/ComfyUI/models/text_encoders
%download https://civitai.red/api/download/models/2945208?fileId=2824387 -o qwen_3_06b_base.safetensors

%cd /content/ComfyUI/models/vae
%download https://civitai.red/api/download/models/2110009?fileId=2004692 -o qwen_image_vae.safetensors

%cd /content/ComfyUI


# Descargar upscalers
print("\n🔄 Verificando modelos de Upscale...")
upscalers = {
    "4x_foolhardy_Remacri.safetensors": "https://huggingface.co/LyliaEngine/4x_foolhardy_Remacri/resolve/main/4x_foolhardy_Remacri.safetensors",
    "4x-AnimeSharp.pth": "https://huggingface.co/Kim2091/AnimeSharp/resolve/main/4x-AnimeSharp.pth",
    "RealESRGAN_x4plus_anime_6B.pth": "https://huggingface.co/gemasai/RealESRGAN_x4plus_anime_6B/resolve/main/RealESRGAN_x4plus_anime_6B.pth",
    "4x_IllustrationJaNai_V1_ESRGAN_135k.pth": "https://huggingface.co/halllooo/4x_illustrationJaNaiV1/resolve/main/4x_IllustrationJaNai_V1_ESRGAN_135k.pth",
    "2xLexicaSwinIR.pth": "https://github.com/Phhofm/models/raw/main/2xLexicaSwinIR/2xLexicaSwinIR.pth",
    "4xmssim_drct-l_pretrain.pth": "https://github.com/Phhofm/models/releases/download/4xDRCT-mssim-pretrains/4xmssim_drct-l_pretrain.pth"
}
for name, url in upscalers.items():
    if not _os.path.exists(f"models/upscale_models/{name}"):
        print(f"   ↳ Adquiriendo {name}...")
        !aria2c --console-log-level=error -c -x 4 -s 4 -k 1M "{url}" -d models/upscale_models -o "{name}"

# 4. Lanzar ComfyUI
print("\n🚀 Lanzando backend de ComfyUI...")
!fuser -k 8188/tcp > /dev/null 2>&1

log_file = open("comfyui.log", "w")
comfy_process = _sp.Popen([
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--preview-method", "latent2rgb",
    "--highvram"
], stdout=log_file, stderr=log_file, start_new_session=True)
clear_output()
# 5. Verificación de estabilidad
print("⏳ Esperando respuesta del servidor web...")
server_ready = False
for i in range(60):
    _time.sleep(1)

    # Comprobar si el proceso murió prematuramente
    if comfy_process.poll() is not None:
        print("\n❌ ¡El servidor de ComfyUI se estrelló durante el arranque!")
        break

    s = _socket.socket(_socket.AF_INET, _socket.SOCK_STREAM)
    try:
        s.connect(("127.0.0.1", 8188))
        s.close()
        server_ready = True
        print("\n============================================================")
        print("✅ ¡Todo Listo!")
        print("============================================================")
        break
    except _socket.error:
        print(".", end="")

if not server_ready:
    print("\n============================================================")
    print("📋 [DIAGNÓSTICO DE CRASH DE COMFYUI]:")
    print("============================================================")
    with open("comfyui.log", "r") as f:
        print(f.read())

## <font color="#e0ae22"> **Anima Base** </font>
- 4+ Cfg, 22+ steps


---
## <font color="#22e077"> **RDBT Anima** </font>
- 1-2 Cfg , 16+ steps
- Mas rapido en generarcion.





In [ ]:
#@title 🖼️ Celda 2: Configurar Parámetros, Upscale y Generar
#@markdown ### La primera vez al generar se demora al cargar los modelos. No se preocupen si ven un "none" cuando generan.
#@markdown ### 💡 Para cancelar la generación en cualquier momento, presiona el botón nativo Detener (⏹️) a la izquierda de esta celda.
#@markdown ---
#@markdown Seleccionar el mismo modelo seleccionado en la celda anterior.
Model= "Anima Base" #@param ["Anima Base","RDBT Anima 1.2"]
if "Anima Base" in Model:
  diffmodel = "anima-base-v1.0.safetensors"
elif "RDBT Anima 1.2" in Model:
  diffmodel = "rdbtAnima_b1V12.safetensors"
#@markdown ---
#@markdown ### Prompts
positive_prompt = "masterpiece, best quality" #@param {type:"string"}
negative_prompt = "worst quality, low quality, signature, watermark, username, blurry, deformed" #@param {type:"string"}

#@markdown ---
#@markdown ### Ajustes de Upscaling (Súper Resolución)
activar_upscale = False #@param {type:"boolean"}
factor_upscale = 1.5 #@param {type:"slider", min:1.0, max:2.0, step:0.05}
modelo_upscale = "4x_foolhardy_Remacri.safetensors" #@param ["4x_foolhardy_Remacri.safetensors", "4x-AnimeSharp.pth", "RealESRGAN_x4plus_anime_6B.pth", "4x_IllustrationJaNai_V1_ESRGAN_135k.pth", "2xLexicaSwinIR.pth", "4xmssim_drct-l_pretrain.pth"]

#@markdown ---
#@markdown ### Configuración del KSampler
cfg_scale = 4.5 #@param {type:"slider", min:1.0, max:20.0, step:0.5}
steps = 24 #@param {type:"slider", min:10, max:100, step:1}
resolution = "832x1216" #@param ["832x1216", "768x1334", "896x1152", "1024x1024","1334x768", "1216x832", "1216x832", "1152x896"]
sampler = "euler_ancestral" #@param ["euler", "euler_ancestral", "dpmpp_2m", "dpmpp_2m_sde_gpu"]
scheduler = "sgm_uniform" #@param ["normal", "karras", "exponential", "sgm_uniform"]
seed = -1 #@param {type:"integer"}

import websocket
import uuid
import json
import urllib.request
import urllib.parse
import urllib.error
import torch
import asyncio
from PIL import Image
import io
from IPython.display import display, clear_output, HTML
from IPython.display import DisplayHandle
from tqdm import tqdm

server_address = "127.0.0.1:8188"
if "client_id" not in globals():
    client_id = str(uuid.uuid4())

width, height = map(int, resolution.replace(",", "x").split("x"))
if seed < 0:
    seed = torch.randint(0, 2**32 - 1, (1,)).item()

workflow_api = {
  "1": { "inputs": { "unet_name": diffmodel, "weight_dtype": "default" }, "class_type": "UNETLoader" },
  "2": { "inputs": { "clip_name": "qwen_3_06b_base.safetensors", "type": "stable_diffusion" }, "class_type": "CLIPLoader" },
  "3": { "inputs": { "vae_name": "qwen_image_vae.safetensors" }, "class_type": "VAELoader" },
  "4": { "inputs": { "text": positive_prompt, "clip": ["2", 0] }, "class_type": "CLIPTextEncode" },
  "5": { "inputs": { "text": negative_prompt, "clip": ["2", 0] }, "class_type": "CLIPTextEncode" },
  "46": { "inputs": { "width": width, "height": height, "batch_size": 1 }, "class_type": "EmptyLatentImage" },
  "7": {
    "inputs": {
      "seed": seed, "steps": steps, "cfg": cfg_scale, "sampler_name": sampler, "scheduler": scheduler,
      "denoise": 1.0, "model": ["1", 0], "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["46", 0]
    },
    "class_type": "KSampler"
  },
  "8": { "inputs": { "samples": ["7", 0], "vae": ["3", 0] }, "class_type": "VAEDecode" }
}

if activar_upscale:
    target_w = int(round(width * factor_upscale))
    target_h = int(round(height * factor_upscale))
    workflow_api["10"] = {
        "inputs": { "model_name": modelo_upscale },
        "class_type": "UpscaleModelLoader"
    }
    workflow_api["11"] = {
        "inputs": { "upscale_model": ["10", 0], "image": ["8", 0] },
        "class_type": "ImageUpscaleWithModel"
    }
    workflow_api["12"] = {
        "inputs": {
            "upscale_method": "lanczos",
            "width": target_w,
            "height": target_h,
            "crop": "disabled",
            "image": ["11", 0]
        },
        "class_type": "ImageScale"
    }
    workflow_api["9"] = {
        "inputs": { "filename_prefix": "Anima_Upscaled", "images": ["12", 0] },
        "class_type": "SaveImage"
    }
else:
    workflow_api["9"] = {
        "inputs": { "filename_prefix": "Anima_Live_Preview", "images": ["8", 0] },
        "class_type": "SaveImage"
    }

def queue_prompt(prompt_workflow):
    p = {"prompt": prompt_workflow, "client_id": client_id}
    data = json.dumps(p).encode('utf-8')
    req = urllib.request.Request(f"http://{server_address}/prompt", data=data)
    try:
        return json.loads(urllib.request.urlopen(req).read().decode('utf-8'))
    except urllib.error.HTTPError as e:
        error_body = e.read().decode('utf-8')
        print("\n❌ [ComfyUI Server Validation Error]:")
        try:
            print(json.dumps(json.loads(error_body), indent=2))
        except:
            print(error_body)
        raise RuntimeError("ComfyUI rechazó la estructura de nodos.")

def get_image(filename, subfolder, folder_type):
    data = {"filename": filename, "subfolder": subfolder, "type": folder_type}
    url_values = urllib.parse.urlencode(data)
    with urllib.request.urlopen(f"http://{server_address}/view?{url_values}") as response:
        return response.read()

def interrupt_and_clear_comfy():
    """Cancela inmediatamente la ejecución actual y limpia toda la cola de peticiones en ComfyUI."""
    try:
        req_int = urllib.request.Request(f"http://{server_address}/interrupt", data=b"{}", method="POST")
        urllib.request.urlopen(req_int, timeout=2)
    except Exception:
        pass

    try:
        data = json.dumps({"clear": True}).encode("utf-8")
        req_clr = urllib.request.Request(
            f"http://{server_address}/queue",
            data=data,
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        urllib.request.urlopen(req_clr, timeout=2)
    except Exception:
        pass

class GeneracionCancelada(Exception):
    pass

async def generate_image_with_preview(prompt_workflow):
    try:
        data = json.dumps({"clear": True}).encode("utf-8")
        req_clr = urllib.request.Request(
            f"http://{server_address}/queue",
            data=data,
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        urllib.request.urlopen(req_clr, timeout=2)
    except Exception:
        pass

    ws = websocket.WebSocket()
    ws.connect(f"ws://{server_address}/ws?clientId={client_id}")

    ws.settimeout(0.1)
    while True:
        try:
            ws.recv()
        except Exception:
            break

    print("🎨 Enviando flujo a ComfyUI...")
    prompt_id = queue_prompt(prompt_workflow)['prompt_id']

    pbar = None
    preview_handle = DisplayHandle()
    preview_handle.display(print("⏳ Esperando primer paso de renderizado..."))

    try:
        while True:
            await asyncio.sleep(0.02)

            ws.settimeout(0.1)
            try:
                out = ws.recv()
            except websocket.WebSocketTimeoutException:
                continue

            if isinstance(out, str):
                message = json.loads(out)
                msg_type = message.get('type')
                msg_data = message.get('data', {})

                if msg_type == 'progress':
                    if pbar is None:
                        pbar = tqdm(
                            total=msg_data['max'],
                            desc="📥 Generando pasos",
                            bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
                        )
                    pbar.n = msg_data['value']
                    pbar.refresh()

                if msg_type == 'execution_interrupted':
                    msg_prompt_id = msg_data.get('prompt_id')
                    if msg_prompt_id is None or msg_prompt_id == prompt_id:
                        raise GeneracionCancelada("La generación fue interrumpida en el backend de ComfyUI.")

                if msg_type == 'execution_error':
                    if msg_data.get('prompt_id') == prompt_id:
                        raise RuntimeError(f"Error de ejecución en nodo: {msg_data.get('node_type')}")

                if msg_type == 'executing':
                    if msg_data.get('node') is None and msg_data.get('prompt_id') == prompt_id:
                        break
            elif isinstance(out, bytes):
                try:
                    image_bytes = out[8:]
                    preview_img = Image.open(io.BytesIO(image_bytes))
                    preview_handle.update(preview_img)
                except Exception:
                    pass
    finally:
        if pbar is not None:
            pbar.close()

    if activar_upscale:
        print(f"🚀 Aplicando súper resolución con {modelo_upscale} (Factor Lanczos: {factor_upscale}x)...")
    else:
        print("✨ Decodificando alta resolución final con el VAE...")

    with urllib.request.urlopen(f"http://{server_address}/history/{prompt_id}") as response:
        history = json.loads(response.read().decode('utf-8'))[prompt_id]

    for node_id in history['outputs']:
        node_output = history['outputs'][node_id]
        if 'images' in node_output:
            for image_info in node_output['images']:
                image_data = get_image(image_info['filename'], image_info['subfolder'], image_info['type'])
                ws.close()
                return Image.open(io.BytesIO(image_data))
    ws.close()
    raise RuntimeError("Error al recuperar la imagen.")

try:
    output_image = await generate_image_with_preview(workflow_api)
    clear_output(wait=True)
    status_msg = f"✅ ¡Inferencia completada con Upscale ({modelo_upscale} @ {factor_upscale}x)!" if activar_upscale else "✅ ¡Inferencia completada exitosamente!"
    print(status_msg)
    print(f"📌 Semilla: {seed} | Dimensión final: {output_image.width}x{output_image.height}")

    output_image.save(f"anima_preview_out_{seed}.png")
    display(output_image)
except (GeneracionCancelada, KeyboardInterrupt):
    interrupt_and_clear_comfy()
    clear_output(wait=True)
    print("🛑 Generación cancelada por el usuario (botón Detener ⏹️ de Colab).")
    print("✅ El backend de ComfyUI sigue activo: puedes ajustar los parámetros y generar de nuevo sin reiniciar nada.")
except Exception as e:
    print(f"\n❌ Detalle del fallo: {e}")